# PocketToons analyst: try it in Colab

An AI analyst agent that investigates *"MAU is up 20% QoQ but revenue is flat. Figure out what's
happening and what we should do about it"* on a synthetic SQLite database, using read-only SQL.
Project, results and design: [README on GitHub](https://github.com/Anshi0202/pockettoons-analyst).

**How to use this notebook:** run the cells in order (Shift+Enter).

- **Steps 1–3** need no API key: get the code, build the database, run the offline tests.
- **Steps 4–7** call the OpenAI API with *your* key, which is never saved in the notebook.
  A simple question costs under $0.01. The full investigation costs about $0.07 and takes 5–8 minutes.

## 1. Get the code and install packages (about 30 seconds)

In [ ]:
import os
if not os.path.exists("/content/pockettoons-analyst"):
    !git clone -q https://github.com/Anshi0202/pockettoons-analyst.git /content/pockettoons-analyst
%cd /content/pockettoons-analyst
!pip -q install openai python-dotenv pyyaml
!git log -1 --format="Code version: %h (%cd)" --date=short

from IPython.display import Markdown, display
def show_md(text):
    # Colab reads a pair of $ signs as a math formula, so show each $ as a plain dollar sign
    display(Markdown(text.replace("$", r"\$")))

## 2. Build the database (about 15 seconds)

`data/generate.py` simulates the app from launch (Oct 2025) and writes April–September 2026 to
`data/pockettoons.db`, with seed 12, so everyone gets the same data. `data/validate.py` then
checks the headline numbers with plain SQL. Expect MAU up about 20% and revenue down about 2%
("flat") as reported.

In [ ]:
!python data/generate.py > /tmp/generate_log.txt && tail -n 1 /tmp/generate_log.txt
!python data/validate.py

## 3. Offline tests (optional, no API key)

These check, in order:
- the agent's safety (read-only SQL) and its loop, against a mock API;
- the eval's grading rules;
- that the improved prompt (v2) adds nothing story-specific;
- that the eval's answer key matches the database.

In [ ]:
!python agent/test_offline.py 2>&1 | tail -n 3
!python eval/test_grader.py | tail -n 1
!python eval/test_prompts.py | tail -n 1
!python eval/check_answer_key.py | tail -n 1

## 4. Add your OpenAI API key

**Option A (recommended):** click the 🔑 key icon in the left sidebar (*Secrets*), add a secret named
`OPENAI_API_KEY` with your key, and switch on *Notebook access*.

**Option B:** run the cell below and paste the key when asked; the input is hidden.

In [ ]:
import os
from getpass import getpass
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Using the OPENAI_API_KEY secret from the sidebar.")
except Exception:
    os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key (hidden): ").strip()
print("Key set:", bool(os.environ.get("OPENAI_API_KEY")))

## 5. Ask the agent a question

Edit the question in the form on the right, then run the cell. The agent prints one line per step
while it works, then writes its answer. Ideas:
- *What was MAU in August 2026?*
- *Which genre had the most views in Q2 2026?*
- *How many paying subscribers did we have at the end of September 2026?* (a trap)

**prompt_version:**
- `v1` is the original prompt.
- `v2` adds four generic method rules from the Phase 4 eval; see `eval/results.md`.

In [ ]:
#@title Ask the agent
question = "What was MAU in August 2026?"  #@param {type:"string"}
prompt_version = "v1"  #@param ["v1", "v2"]
max_queries = 40  #@param {type:"integer"}

import os, sys, json
from IPython.display import Markdown, display
sys.path.insert(0, "/content/pockettoons-analyst")
os.environ["PT_PROMPT"] = prompt_version
from agent.agent import run_agent

res = run_agent(question, max_queries=max_queries)
show_md(res["answer"])
print("\n" + " | ".join(f"{k}: {v}" for k, v in res["stats"].items()))

## 6. See what the agent did, step by step

The agent logs every step: why it ran each query, how many rows came back, and its hypothesis board.
This shows the last run from step 5 or step 7. The second cell shows one query's SQL and result.

In [ ]:
#@title Steps of the last run
import json
import pandas as pd
from IPython.display import display

def steps_table(trace_jsonl):
    rows = []
    for line in open(trace_jsonl, encoding="utf-8"):
        e = json.loads(line)
        if e.get("kind") != "tool":
            continue
        if e["name"] == "run_sql":
            what = e.get("purpose", "")
            result = f"{e.get('rows')} rows" if e.get("ok") else f"error: {e.get('error')}"
        elif e["name"] == "update_hypotheses":
            what = " · ".join(f"[{h.get('status')}] {h.get('statement')}" for h in e.get("hypotheses", []))
            result = ""
        else:
            what, result = "read the database schema", ""
        rows.append({"step": e.get("step"), "tool": e["name"], "what and why": what, "result": result})
    return pd.DataFrame(rows).set_index("step")

pd.set_option("display.max_colwidth", 250)
display(steps_table(res["trace_jsonl"]))
print("Full trace (Markdown):", res["trace_md"])

In [ ]:
#@title Look at one query: its SQL and result
step = 2  #@param {type:"integer"}
from IPython.display import Markdown, display
found = False
for line in open(res["trace_jsonl"], encoding="utf-8"):
    e = json.loads(line)
    if e.get("kind") == "tool" and e.get("step") == step and e["name"] == "run_sql":
        found = True
        show_md(f"**Purpose:** {e.get('purpose')}\n\n```sql\n{e['query'].strip()}\n```\n\n{e['output']}")
if not found:
    print(f"Step {step} is not a SQL query. Pick a step with tool = run_sql from the table above.")

## 7. The GM's question (5–8 minutes, about $0.07)

This is the open investigation the project is built around. The answer has the same sections each
time: answer, definitions, data quality, findings, rejected hypotheses, recommendations and
confidence. Answers vary from run to run; the eval in step 8 and `eval/results.md` measure that.
After it finishes, run the step-6 cells again to see its 30–40 steps.

In [ ]:
#@title Run the GM's question
prompt_version = "v2"  #@param ["v1", "v2"]
os.environ["PT_PROMPT"] = prompt_version
gm_question = ("MAU is up 20% QoQ but revenue is flat. Board meeting Thursday. "
               "Figure out what's happening and what we should do about it.")
res = run_agent(gm_question)
show_md(res["answer"])
print("\n" + " | ".join(f"{k}: {v}" for k, v in res["stats"].items()))

## 8. A short eval (6 questions, about 2 minutes, about 1 cent)

Six of the 18 eval questions:
- two plain lookups;
- two data traps;
- two questions the agent must decline (one asks it to delete data).

Each answer is graded automatically against an answer key computed with SQL. The runner also checks
that the database file is unchanged afterwards.

In [ ]:
#@title Run the short eval
prompt_version = "v1"  #@param ["v1", "v2"]
!python eval/run_eval.py --label colab_{prompt_version} --env PT_PROMPT={prompt_version} --only L1_signups_aug L3_top_genre_q2 T1_paying_subs_sep T3_us_users_pre2026 U2_nps U3_delete_affiliate
import glob
from IPython.display import Markdown, display
latest = sorted(glob.glob(f"eval/runs/*_colab_{prompt_version}"))[-1]
show_md(open(f"{latest}/summary.md", encoding="utf-8").read())

## 9. The project's own results (no API calls)

The full eval write-up from the repository: baseline vs v2, three runs each, plus the control test.

In [ ]:
from IPython.display import Markdown, display
show_md(open("eval/results.md", encoding="utf-8").read())

### Spoiler: the answer key

`data/story.md` describes what was planted in the data. The agent never sees this file.
Run the cell only if you want the answers.

In [ ]:
from IPython.display import Markdown, display
show_md(open("data/story.md", encoding="utf-8").read())